# Problem Statement
This project uses the 2024 NYC Yellow Taxi Trip Records, published by the New York City Taxi and Limousine Commission (TLC). The dataset contains 19 columns describing taxi journeys, including trip timestamps, distance, fares, pickup and drop-off locations, and payment type.
The objective is to predict the payment_type column using four classification algorithms: Logistic Regression, Random Forest, Gradient Boosting, and XGBoost. PySpark will be used for data cleaning, feature engineering, model training, hyperparameter tuning, and evaluation using suitable performance metrics. The project will also investigate model explainability, stability, and Spark performance, with results presented through four Tableau dashboards.

# Dataset Provenance and Licence Evidence

The dataset used in this project is the NYC Yellow Taxi Trip Records for January–December 2024, published by the New York City Taxi and Limousine Commission (TLC). The official TLC Trip Record Data website provides monthly trip records in Parquet format, containing information such as pickup and drop-off timestamps, trip distances, locations, passenger counts, and fare amounts.

Dataset details:
- Dataset name: NYC Yellow Taxi Trip Records (2024)
- Publisher: New York City Taxi and Limousine Commission (TLC)
- Source URL: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
- Period covered: January–December 2024
- Format: Monthly Parquet files
- Processing: The twelve monthly files were combined using PySpark.
- Target variable: payment_type
- Provenance evidence: The original downloaded filenames, source URL, download date, and all twelve monthly filenames should be recorded to establish the dataset's origin and coverage.
- Licence and ethical considerations: The NYC Yellow Taxi Trip Records for 2024 were obtained from the official NYC Taxi and Limousine Commission website. The corresponding NYC Open Data record lists the licence as unspecified. Therefore, no specific open-source licence is claimed. The applicable NYC Open Data and TLC usage terms should be reviewed to establish suitability for academic use.

In [ ]:
# importing required libraries
from pyspark.sql import SparkSession
import zipfile
import os
import pandas as pd

In [ ]:
# extracting the zip file
zip_path = "/content/data.zip"
extract_path = "/content/extract_data"
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    bad_file = zip_ref.testzip()
    if bad_file is not None:
        raise ValueError(f"corrupted file: {bad_file}")
    zip_ref.extractall(extract_path)
print("zip extraction completed successfully")

zip extraction completed successfully


In [ ]:
# detecting the all month parquet file present in folder
extract_path = "/content/extract_data/data"
parquet_files = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(".parquet"):
            parquet_files.append(os.path.join(root, file))
parquet_files.sort()
print("Total Parquet files:", len(parquet_files))
for file in parquet_files:
    print(os.path.basename(file))

Total Parquet files: 12
yellow_tripdata_2024-01.parquet
yellow_tripdata_2024-02.parquet
yellow_tripdata_2024-03.parquet
yellow_tripdata_2024-04.parquet
yellow_tripdata_2024-05.parquet
yellow_tripdata_2024-06.parquet
yellow_tripdata_2024-07.parquet
yellow_tripdata_2024-08.parquet
yellow_tripdata_2024-09.parquet
yellow_tripdata_2024-10.parquet
yellow_tripdata_2024-11.parquet
yellow_tripdata_2024-12.parquet


In [ ]:
# creating the spark session
spark = (SparkSession.builder.appName("CombineNYCTaxi2024").getOrCreate())

In [ ]:
# combine all monthly parquet files
combined_df = spark.read.parquet(*parquet_files)

In [ ]:
# schema inspection
combined_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [ ]:
# check the no of rows and columns
print("total rows:", combined_df.count())
print("total columns:", len(combined_df.columns))
combined_df.show(20, truncate=False)

total rows: 41169720
total columns: 19
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-10-01 00:30:44 |2024-10-01 00:48:26  |1              |3.0          |1         |N                 |162         |246         |1        

In [ ]:
# verify dataset dimensions
total_rows = combined_df.count()
total_columns = len(combined_df.columns)
if total_rows >= 10_000_000 and total_columns >= 10:
    print("row and column requirements satisfied.")
else:
    print("dataset requirements not satisfied.")

row and column requirements satisfied.


In [ ]:
# save the combined dataset
output_path = "/content/combined_yellow_taxi_2024"
(combined_df.write.mode("overwrite").parquet(output_path))
print("combined dataset saved successfully:", output_path)

combined dataset saved successfully: /content/combined_yellow_taxi_2024


In [ ]:
# combine data the last 20 rows
last_20_rows = combined_df.tail(20)
display(pd.DataFrame(last_20_rows, columns=combined_df.columns))

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,1,2024-01-31 23:22:09,2024-01-31 23:33:41,None,2.30,None,None,211,170,0,12.10,1.00,0.5,0.00,0.00,1.0,17.10,None,None
1,1,2024-01-31 23:02:28,2024-01-31 23:19:20,None,2.00,None,None,161,239,0,14.90,1.00,0.5,2.98,0.00,1.0,22.88,None,None
2,2,2024-01-31 23:33:02,2024-01-31 23:43:41,None,1.66,None,None,107,246,0,12.34,0.00,0.5,1.63,0.00,1.0,17.97,None,None
3,1,2024-01-31 23:01:47,2024-01-31 23:08:51,None,1.30,None,None,158,234,0,9.30,1.00,0.5,2.15,0.00,1.0,16.45,None,None
4,2,2024-01-31 23:46:39,2024-01-31 23:57:31,None,3.27,None,None,161,24,0,15.73,0.00,0.5,0.00,0.00,1.0,19.73,None,None
5,1,2024-01-31 23:40:27,2024-01-31 23:54:56,None,9.00,None,None,138,140,0,34.50,7.75,0.5,7.97,6.94,1.0,61.16,None,None
6,2,2024-01-31 23:49:00,2024-02-01 00:08:00,None,4.30,None,None,48,42,0,23.32,0.00,0.5,5.46,0.00,1.0,32.78,None,None
7,1,2024-01-31 23:01:28,2024-01-31 23:20:46,None,4.00,None,None,255,90,0,21.91,0.00,0.5,0.00,0.00,1.0,25.91,None,None
8,1,2024-01-31 23:52:57,2024-01-31 23:56:48,None,0.70,None,None,230,48,0,5.10,1.00,0.5,1.51,0.00,1.0,11.61,None,None
9,2,2024-01-31 23:16:55,2024-01-31 23:57:34,None,28.19,None,None,68,265,0,88.24,0.00,0.5,20.36,6.36,1.0,118.96,None,None


In [ ]:
# calculate the original zip file size in bytes
zip_size_bytes = os.path.getsize(zip_path)
# calculate the combined size of the monthly Parquet files
parquet_size_bytes = sum(os.path.getsize(file_path) for file_path in parquet_files)

In [ ]:
# convert bytes to GB
zip_size_gb = zip_size_bytes / (1000 ** 3)
parquet_size_gb = parquet_size_bytes / (1000 ** 3)
print(f"ZIP file size: {zip_size_gb:.3f} GB")
print(f"total monthly Parquet size: {parquet_size_gb:.3f} GB")

ZIP file size: 0.682 GB
total monthly Parquet size: 0.693 GB


In [ ]:
# check the dataset size against the 1 GB threshold
if parquet_size_bytes < 1_000_000_000:
    print("dataset size: Less than 1 GB")
else:
    print("dataset size: At least 1 GB")

dataset size: Less than 1 GB


In [ ]:
combined_check = spark.read.parquet(output_path)
print("Saved dataset schema:")
combined_check.printSchema()
spark.catalog.clearCache()

Saved dataset schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



###  Five V’s of Big Data

- **Volume:** The dataset contains a large number of taxi-trip records. The notebook measures the actual row count and number of columns to check the coursework's minimum requirements of 10 million rows and 10 columns.
- **Velocity:** Taxi trips are generated continuously during real-world operations. However, this project analyses historical 2024 records rather than processing live taxi data, so it does not demonstrate real-time ingestion.
- **Variety:** The dataset contains structured fields with different data types, such as timestamps, numerical trip measurements, and location identifiers. This provides different types of information for analysis, although the dataset is primarily structured rather than unstructured.
- **Veracity:** Trip records may contain missing, invalid, or implausible values. These issues need to be investigated during preprocessing before reliable modelling can be performed.
- **Value:** Predicting payment type may help identify patterns in how taxi trips are paid for. The usefulness of these predictions will be assessed through classification metrics, while recognising the limitations of the dataset.